# 🖥️ SWASH tutorial

This tutorial shows how to load in native Matlab files from the SWASH model and convert them to a Parcels-compatible `FieldSet`. The tutorial also shows how to run a simple particle tracking simulation using the SWASH data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import parcels
import parcels.tutorial

We have prepared a swash simulation based on the wave flume experiment in [Alsina et al., 2020](https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2020JC016294). We have then extracted a subset of this flowfield data, which is available in `.mat` format. 

Because of these `.mat` files, the APi for the `parcels.convert.swash_to_sgrid()` function requires a `field_files` dictionary to correctly map the SWASH output files to the corresponding field names.

In [ ]:
files = parcels.tutorial.get_dataset_files("SWASH_data/data")

names = ["coord", "U", "V", "W", "WATLEV", "ZK"]
file_dict = {name: file for name, file in zip(names, files, strict=True)}

ds = parcels.convert.swash_to_sgrid(field_files=file_dict)
display(ds)

To make the simulation "cleaner", we select only one grid cell in the Y-direction. We also set the meridional velocity component to zero to focus only on the movement in the X-Z plane. Note that you would not do this on a three-dimensional simulation.

In [ ]:
# select only one grid cell in the Y-direction and update the grid face dimensions accordingly
ds = ds.isel(YG=slice(0, 2), YC=slice(0, 1))
ds["grid"].attrs["face_dimensions"] = "XC:XG (padding:high) YC:YG (padding:none)"

# set the meridional velocity component to zero
ds["V"].data[:] = 0.0

We can now use this new `DataSet` to create a `FieldSet` for our Parcels simulation.

In [ ]:
fieldset = parcels.FieldSet.from_sgrid_conventions(ds)
fieldset.describe()

For the simulation, we also need an extra kernel that puts particles that move into the air back onto the water surface.

In [ ]:
def KeepInOcean(particles, fieldset):
    # find particles that move through the surface
    through_surface = particles.state == parcels.StatusCode.ErrorOutOfBounds
    eta = fieldset.watlev[particles[through_surface]]
    particles[through_surface].dz = eta - particles[through_surface].z
    particles[through_surface].state = parcels.StatusCode.Success

Now, we can use the `FieldSet` and `KeepInOcean` kernel to run a simulation

In [ ]:
x, z = np.meshgrid(np.linspace(23.5, 24.75, 10), np.linspace(-0.004, 0.002, 3))
y = np.full_like(x, 0.0)

pset = parcels.ParticleSet(fieldset, z=z, y=y, x=x)

output_file = parcels.ParticleFile(
    "output-swash.parquet", outputdt=np.timedelta64(25, "ms"), mode="w"
)
pset.execute(
    [parcels.kernels.AdvectionRK2_3D, KeepInOcean],
    runtime=np.timedelta64(3, "s"),
    dt=np.timedelta64(5, "ms"),
    output_file=output_file,
    verbose_progress=False,
)

And then make an animation of the simulation output, along with the water level at each time step.

In [ ]:
import matplotlib as mpl
import polars as pl
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection

# for interactive display of animation
plt.rcParams["animation.html"] = "jshtml"

df = parcels.read_particlefile("output-swash.parquet")

time_step = np.timedelta64(25, "ms")  # time step for animation frames

# Build the time axis for the animation.
timerange = np.arange(
    np.nanmin(df["t"]),
    np.nanmax(df["t"]) + time_step,
    time_step,
)

# figure setup
fig, ax = plt.subplots(figsize=(8, 4))
cmap = mpl.colormaps["tab20b"]
X = fieldset.U.grid.lon
ax.set_xlim(X.min(), X.max())
ax.set_ylim(-0.04, 0.03)

# set up the water level
(line,) = ax.plot([], [], lw=1, color="navy")
trails = LineCollection([], linewidths=0.6, alpha=0.3)
ax.add_collection(trails)
plt.plot(X, -fieldset.botlev.data[0, 0, 0, :], color="k")  ## bottom


# Convert the trajectory data to a time-by-particle array and assign colors.
particle_ids = df["particle_id"].unique().to_list()
colors = np.asarray(
    [cmap(i / max(len(particle_ids) - 1, 1)) for i in range(len(particle_ids))]
)

# Create arrays to hold the x and z positions of each particle at each time step.
x = np.full((len(particle_ids), len(timerange)), np.nan)
z = np.full((len(particle_ids), len(timerange)), np.nan)

traj = df.with_columns(
    pl.col("particle_id")
    .replace(particle_ids, range(len(particle_ids)))
    .alias("p_idx"),
    pl.int_range(pl.len()).alias("row_idx"),
)

# Map each observation to its nearest animation time index.
traj = traj.with_columns(
    pl.Series("t_idx", np.searchsorted(timerange, traj["t"].to_numpy())).alias("t_idx")
)

x[traj["p_idx"].to_numpy(), traj["t_idx"].to_numpy()] = traj["x"].to_numpy()
z[traj["p_idx"].to_numpy(), traj["t_idx"].to_numpy()] = traj["z"].to_numpy()

# Plot first timestep
scatter = ax.scatter(x[:, 0], z[:, 0], s=10, c=colors, zorder=2)

# Set initial title
t_str = f"{timerange[0] / np.timedelta64(1, 's'):.3f} s"
title = ax.set_title(f"Particles at t = {t_str}")


def animate(i):
    h = fieldset.watlev.data[i, 0, 0, :]
    line.set_data(X, h)

    t_str = f"{timerange[i] / np.timedelta64(1, 's'):.3f} s"
    title.set_text(f"Particles at t = {t_str}")

    scatter.set_offsets(np.column_stack((x[:, i], z[:, i])))

    # keep full trail of particle positions
    x_slice = x[:, : i + 1]
    z_slice = z[:, : i + 1]
    trails.set_segments(np.dstack((x_slice, z_slice)))
    trails.set_color(colors)


# Create animation
anim = FuncAnimation(fig, animate, frames=len(timerange), interval=100)
plt.close(fig)
anim